In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

file = r"/Users/aminesehairi/Desktop/quant learning material/DATA BOOK DEPTH /2026-08-19_15-03-57/processed_microstructure/BTC_ETH_100ms.parquet"

market = pd.read_parquet(file)

market.shape

(241111, 54)

In [15]:
import numpy as np
import pandas as pd

# ============================================================
# 1. LOAD FULL BTC 100ms BOOK
# ============================================================

btc_path = r"/Users/aminesehairi/Desktop/quant learning material/DATA BOOK DEPTH /2026-08-19_15-03-57/processed_microstructure/BTC_100ms.parquet"

btc_full = pd.read_parquet(btc_path)

print("Full BTC shape:", btc_full.shape)
print("Market shape:", market.shape)

print("\nBTC date range:")
print(btc_full.index.min(), "->", btc_full.index.max())

print("\nMarket date range:")
print(market.index.min(), "->", market.index.max())


# ============================================================
# 2. CORRECT L1-L20 COLUMN NAMES
# ============================================================

bid_qty_cols_raw = [f"bid_qty_{i}" for i in range(1, 21)]
ask_qty_cols_raw = [f"ask_qty_{i}" for i in range(1, 21)]

required_book_cols = bid_qty_cols_raw + ask_qty_cols_raw

missing = [c for c in required_book_cols if c not in btc_full.columns]

if missing:
    raise ValueError(
        f"Still missing some L1-L20 columns: {missing}"
    )

print("\nAll L1-L20 quantity columns found.")


# ============================================================
# 3. EXTRACT + RENAME TO BTC PREFIX
# ============================================================




# CELL 3
# CONFIG
EPS = 1e-12
EVENT_HALFLIFE = 20
SIGNAL_LAG = 1
RATE_WINDOW_1S = 10
MOMENTUM_FAST_WINDOW = 5
MOMENTUM_FAST_SECONDS = 0.5
MOMENTUM_SLOW_WINDOW = 20
MOMENTUM_SLOW_SECONDS = 2.0

# CELL 4
# all feature-building code


# ============================================================
# 4. MERGE ONTO MARKET
# ============================================================

market_full = market.join(
    book20,
    how="left"
)

print("\nMerged shape:", market_full.shape)

print(
    "Total missing L1-L20 values after merge:",
    market_full[book20.columns].isna().sum().sum()
)

# ============================================================
# 5. WORKING COPY
# ============================================================

df = market_full.copy().sort_index()


# ============================================================
# 6. CORE SERIES
# ============================================================

mid = df["btc_mid_price"].astype(float)
spread = df["btc_spread"].astype(float)

bid = mid - spread / 2
ask = mid + spread / 2

bid_qty_l1 = df["btc_bid_depth_1"].astype(float)
ask_qty_l1 = df["btc_ask_depth_1"].astype(float)

buy_qty = df["btc_buy_qty"].fillna(0.0).astype(float)
sell_qty = df["btc_sell_qty"].fillna(0.0).astype(float)

gap = df["btc_data_gap"].fillna(True).astype(bool)

previous_gap = gap.shift(1, fill_value=True)

valid_pair = (~gap) & (~previous_gap)


# ============================================================
# 7. SEGMENT ID
# Prevent signals crossing gaps
# ============================================================

break_flag = gap | previous_gap

segment_id = break_flag.astype(int).cumsum()


# ============================================================
# 8. HELPERS
# ============================================================

def event_conditioned_ema(series, halflife=20, neutral=0.5):
    """
    Event-conditioned EMA.

    NaN = no event -> state not updated.
    Shift(1) prevents current event outcome entering current signal.
    """
    
    result = (
        series
        .groupby(segment_id)
        .transform(
            lambda s:
            s.ewm(
                halflife=halflife,
                adjust=False,
                ignore_na=True
            )
            .mean()
            .shift(1)
        )
    )

    return result.fillna(neutral).clip(0, 1)


def segmented_shift(series, periods=1):
    
    return (
        series
        .groupby(segment_id)
        .shift(periods)
    )


def segmented_rolling_sum(series, window, min_periods=None):

    if min_periods is None:
        min_periods = window

    return (
        series
        .groupby(segment_id)
        .transform(
            lambda s:
            s.rolling(
                window=window,
                min_periods=min_periods
            )
            .sum()
        )
    )


# ============================================================
# 9. STANDARD MICROPRICE
# ============================================================

# Use the already-computed microprice from your master table.

df["standard_microprice"] = df["btc_microprice"]

df["standard_microprice_edge_bps"] = (
    10_000
    *
    (
        df["standard_microprice"]
        - mid
    )
    /
    mid
)


# ============================================================
# 10. PREVIOUS L1 STATE
# ============================================================

previous_bid = bid.shift(1)
previous_ask = ask.shift(1)

previous_bid_qty = bid_qty_l1.shift(1)
previous_ask_qty = ask_qty_l1.shift(1)


same_bid = pd.Series(
    np.isclose(
        bid.to_numpy(),
        previous_bid.to_numpy(),
        atol=1e-9,
        rtol=0,
        equal_nan=False
    ),
    index=df.index
)


same_ask = pd.Series(
    np.isclose(
        ask.to_numpy(),
        previous_ask.to_numpy(),
        atol=1e-9,
        rtol=0,
        equal_nan=False
    ),
    index=df.index
)


# ============================================================
# 11. ATTACK EVENTS
# ============================================================

ask_hit = (
    (buy_qty > 0)
    &
    valid_pair
)

bid_hit = (
    (sell_qty > 0)
    &
    valid_pair
)


# ============================================================
# 12. HIT-AND-HOLD
# ============================================================

ask_hold_ratio = (
    ask_qty_l1
    /
    previous_ask_qty.replace(0, np.nan)
).clip(0, 1)


bid_hold_ratio = (
    bid_qty_l1
    /
    previous_bid_qty.replace(0, np.nan)
).clip(0, 1)


ask_hit_hold_obs = pd.Series(
    np.nan,
    index=df.index,
    dtype=float
)

bid_hit_hold_obs = pd.Series(
    np.nan,
    index=df.index,
    dtype=float
)


# Ask attacked and survives
mask = (
    ask_hit
    &
    same_ask
    &
    (previous_ask_qty > 0)
)

ask_hit_hold_obs.loc[mask] = (
    ask_hold_ratio.loc[mask]
)


# Ask disappears
mask = (
    ask_hit
    &
    (~same_ask)
    &
    (previous_ask_qty > 0)
)

ask_hit_hold_obs.loc[mask] = 0.0


# Bid attacked and survives
mask = (
    bid_hit
    &
    same_bid
    &
    (previous_bid_qty > 0)
)

bid_hit_hold_obs.loc[mask] = (
    bid_hold_ratio.loc[mask]
)


# Bid disappears
mask = (
    bid_hit
    &
    (~same_bid)
    &
    (previous_bid_qty > 0)
)

bid_hit_hold_obs.loc[mask] = 0.0


df["bid_hit_hold"] = event_conditioned_ema(
    bid_hit_hold_obs,
    EVENT_HALFLIFE
)

df["ask_hit_hold"] = event_conditioned_ema(
    ask_hit_hold_obs,
    EVENT_HALFLIFE
)


df["hit_hold_asymmetry"] = (
    df["bid_hit_hold"]
    -
    df["ask_hit_hold"]
)


# ============================================================
# 13. REPLENISHMENT
# ============================================================

# ---------- ASK ----------

ask_consumed = pd.concat(
    [
        buy_qty.rename("flow"),
        previous_ask_qty.rename("depth")
    ],
    axis=1
).min(
    axis=1,
    skipna=False
)


ask_expected_remaining = (
    previous_ask_qty
    -
    buy_qty
).clip(lower=0)


ask_replenished = (
    ask_qty_l1
    -
    ask_expected_remaining
).clip(lower=0)


ask_replenishment_ratio = (
    ask_replenished
    /
    ask_consumed.replace(0, np.nan)
).clip(0, 1)


ask_replenishment_obs = pd.Series(
    np.nan,
    index=df.index,
    dtype=float
)


mask = (
    ask_hit
    &
    same_ask
    &
    (previous_ask_qty > 0)
)

ask_replenishment_obs.loc[mask] = (
    ask_replenishment_ratio.loc[mask]
)


mask = (
    ask_hit
    &
    (~same_ask)
    &
    (previous_ask_qty > 0)
)

ask_replenishment_obs.loc[mask] = 0.0


# ---------- BID ----------

bid_consumed = pd.concat(
    [
        sell_qty.rename("flow"),
        previous_bid_qty.rename("depth")
    ],
    axis=1
).min(
    axis=1,
    skipna=False
)


bid_expected_remaining = (
    previous_bid_qty
    -
    sell_qty
).clip(lower=0)


bid_replenished = (
    bid_qty_l1
    -
    bid_expected_remaining
).clip(lower=0)


bid_replenishment_ratio = (
    bid_replenished
    /
    bid_consumed.replace(0, np.nan)
).clip(0, 1)


bid_replenishment_obs = pd.Series(
    np.nan,
    index=df.index,
    dtype=float
)


mask = (
    bid_hit
    &
    same_bid
    &
    (previous_bid_qty > 0)
)

bid_replenishment_obs.loc[mask] = (
    bid_replenishment_ratio.loc[mask]
)


mask = (
    bid_hit
    &
    (~same_bid)
    &
    (previous_bid_qty > 0)
)

bid_replenishment_obs.loc[mask] = 0.0


# Event-conditioned states

df["bid_replenishment"] = event_conditioned_ema(
    bid_replenishment_obs,
    EVENT_HALFLIFE
)

df["ask_replenishment"] = event_conditioned_ema(
    ask_replenishment_obs,
    EVENT_HALFLIFE
)


df["replenishment_asymmetry"] = (
    df["bid_replenishment"]
    -
    df["ask_replenishment"]
)


# ============================================================
# 14. REPLENISHMENT-ADJUSTED MICROPRICE
# ============================================================

effective_bid_qty = (
    bid_qty_l1
    *
    df["bid_replenishment"]
)

effective_ask_qty = (
    ask_qty_l1
    *
    df["ask_replenishment"]
)


effective_total = (
    effective_bid_qty
    +
    effective_ask_qty
).replace(0, np.nan)


df["replenishment_microprice"] = (
    bid * effective_ask_qty
    +
    ask * effective_bid_qty
) / effective_total


df["replenishment_edge_bps"] = (
    10_000
    *
    (
        df["replenishment_microprice"]
        -
        mid
    )
    /
    mid
)


# ============================================================
# 15. FULL L1-L20 FRACTIONAL PENETRATION
# ============================================================

bid_book = df[bid_qty_cols].astype(float)
ask_book = df[ask_qty_cols].astype(float)

# IMPORTANT:
# aggressive trades at time t are evaluated against the book from t-1.

previous_ask_book = ask_book.shift(1)
previous_bid_book = bid_book.shift(1)


def fractional_penetration(
    flow_series,
    previous_book,
    valid_mask
):

    flow = flow_series.to_numpy(dtype=float)
    q = previous_book.to_numpy(dtype=float)

    valid_book = (
        np.isfinite(q).all(axis=1)
        &
        (q >= 0).all(axis=1)
    )

    valid = (
        valid_mask.to_numpy()
        &
        valid_book
        &
        np.isfinite(flow)
    )


    # Quantity resting before each level.
    cumulative_before = (
        np.cumsum(q, axis=1)
        -
        q
    )


    # Flow available when arriving at each level.
    remaining = (
        flow[:, None]
        -
        cumulative_before
    )


    fractions = np.zeros_like(
        q,
        dtype=float
    )


    positive_depth = q > 0


    np.divide(
        remaining,
        q,
        out=fractions,
        where=positive_depth
    )


    fractions = np.clip(
        fractions,
        0,
        1
    )


    penetration = fractions.sum(axis=1)


    total_depth20 = np.nansum(
        q,
        axis=1
    )


    overflow = (
        flow > total_depth20
    )


    excess_qty = np.maximum(
        flow - total_depth20,
        0
    )


    penetration[~valid] = np.nan
    excess_qty[~valid] = np.nan
    total_depth20[~valid] = np.nan

    overflow = (
        overflow
        &
        valid
    )


    return (
        pd.Series(
            penetration,
            index=df.index
        ),

        pd.Series(
            overflow,
            index=df.index
        ),

        pd.Series(
            excess_qty,
            index=df.index
        ),

        pd.Series(
            total_depth20,
            index=df.index
        )
    )


# BUY attacks ASK
(
    df["buy_penetration"],
    df["buy_depth20_overflow"],
    df["buy_depth20_excess_qty"],
    df["previous_ask_depth20"]

) = fractional_penetration(
    buy_qty,
    previous_ask_book,
    valid_pair
)


# SELL attacks BID
(
    df["sell_penetration"],
    df["sell_depth20_overflow"],
    df["sell_depth20_excess_qty"],
    df["previous_bid_depth20"]

) = fractional_penetration(
    sell_qty,
    previous_bid_book,
    valid_pair
)


# ============================================================
# 16. IMMEDIATE PENETRATION
# ============================================================

df["penetration_immediate"] = (
    df["buy_penetration"]
    -
    df["sell_penetration"]
)


# ============================================================
# 17. VOLUME-WEIGHTED PENETRATION
# ============================================================

penetration_volume_numerator = (
    buy_qty
    *
    df["buy_penetration"]
    -
    sell_qty
    *
    df["sell_penetration"]
)


total_aggressive_volume = (
    buy_qty
    +
    sell_qty
)


df["penetration_volume"] = np.where(
    total_aggressive_volume > 0,

    penetration_volume_numerator
    /
    (
        total_aggressive_volume
        +
        EPS
    ),

    0.0
)


df["penetration_volume"] = pd.Series(
    df["penetration_volume"],
    index=df.index
).where(
    valid_pair
)


# ============================================================
# 18. 1-SECOND PENETRATION RATE
# ============================================================

penetration_sum_1s = segmented_rolling_sum(
    df["penetration_immediate"],
    RATE_WINDOW_1S,
    RATE_WINDOW_1S
)


df["penetration_rate_1s"] = (
    penetration_sum_1s
)


# ============================================================
# 19. PENETRATION MOMENTUM
# ============================================================

fast_sum = segmented_rolling_sum(
    df["penetration_immediate"],
    MOMENTUM_FAST_WINDOW,
    MOMENTUM_FAST_WINDOW
)


slow_sum = segmented_rolling_sum(
    df["penetration_immediate"],
    MOMENTUM_SLOW_WINDOW,
    MOMENTUM_SLOW_WINDOW
)


df["penetration_rate_fast"] = (
    fast_sum
    /
    MOMENTUM_FAST_SECONDS
)


df["penetration_rate_slow"] = (
    slow_sum
    /
    MOMENTUM_SLOW_SECONDS
)


df["penetration_momentum"] = (
    df["penetration_rate_fast"]
    -
    df["penetration_rate_slow"]
)


# ============================================================
# 20. KEEP EXISTING OBI / FLOW FEATURES
# ============================================================

df["obi_1"] = df["btc_obi_1"]
df["obi_5"] = df["btc_obi_5"]
df["obi_20"] = df["btc_obi_20"]

df["weighted_obi_20"] = (
    df["btc_weighted_obi_20"]
)

df["trade_pressure"] = (
    df["btc_trade_pressure"]
)


# ============================================================
# 21. CREATE FINAL 100ms CAUSAL SIGNALS
# ============================================================

RAW_FEATURES = {

    "standard_microprice":
        "standard_microprice_edge_bps",

    "replenishment_microprice":
        "replenishment_edge_bps",

    "hit_hold_asymmetry":
        "hit_hold_asymmetry",

    "replenishment_asymmetry":
        "replenishment_asymmetry",

    "penetration_immediate":
        "penetration_immediate",

    "penetration_volume":
        "penetration_volume",

    "penetration_rate_1s":
        "penetration_rate_1s",

    "penetration_momentum":
        "penetration_momentum",

    "obi_1":
        "obi_1",

    "obi_5":
        "obi_5",

    "obi_20":
        "obi_20",

    "weighted_obi_20":
        "weighted_obi_20",

    "trade_pressure":
        "trade_pressure",
}


for feature_name, raw_col in RAW_FEATURES.items():

    df[f"signal_{feature_name}"] = (
        segmented_shift(
            df[raw_col],
            SIGNAL_LAG
        )
    )


# ============================================================
# 22. INVALIDATE SIGNALS AT DATA GAPS
# ============================================================

signal_cols = [
    c
    for c in df.columns
    if c.startswith("signal_")
]


df.loc[
    gap,
    signal_cols
] = np.nan


# ============================================================
# 23. FINAL RESEARCH DATAFRAME
# ============================================================

final_features = [

    # Market state
    "btc_mid_price",
    "btc_spread",

    # Standard MP
    "standard_microprice",
    "standard_microprice_edge_bps",

    # Hit hold
    "bid_hit_hold",
    "ask_hit_hold",
    "hit_hold_asymmetry",

    # Replenishment
    "bid_replenishment",
    "ask_replenishment",
    "replenishment_asymmetry",
    "replenishment_microprice",
    "replenishment_edge_bps",

    # Penetration
    "buy_penetration",
    "sell_penetration",
    "penetration_immediate",
    "penetration_volume",
    "penetration_rate_1s",
    "penetration_rate_fast",
    "penetration_rate_slow",
    "penetration_momentum",

    # Existing OBI
    "obi_1",
    "obi_5",
    "obi_20",
    "weighted_obi_20",
    "trade_pressure",

    # Final causal signals
    "signal_standard_microprice",
    "signal_replenishment_microprice",
    "signal_hit_hold_asymmetry",
    "signal_replenishment_asymmetry",
    "signal_penetration_immediate",
    "signal_penetration_volume",
    "signal_penetration_rate_1s",
    "signal_penetration_momentum",
    "signal_obi_1",
    "signal_obi_5",
    "signal_obi_20",
    "signal_weighted_obi_20",
    "signal_trade_pressure",
]


research_df = df[final_features].copy()


# ============================================================
# 24. FINAL CHECKS
# ============================================================

print("=" * 100)
print("FULL HFT FEATURE BUILD COMPLETE")
print("=" * 100)

print("Rows:", f"{len(research_df):,}")
print("Columns:", research_df.shape[1])

print("\nRange:")
print(
    research_df.index.min(),
    "->",
    research_df.index.max()
)

print("\nFinal causal signals:")

for c in signal_cols:
    print(" ", c)


print("\nMissing fractions:")

display(
    research_df
    .isna()
    .mean()
    .sort_values(
        ascending=False
    )
    .to_frame(
        "missing_fraction"
    )
)


print("\nPreview:")

display(
    research_df.head(20)
)

Full BTC shape: (241111, 107)
Market shape: (241111, 54)

BTC date range:
2026-08-19 15:30:27+00:00 -> 2026-08-19 22:12:18+00:00

Market date range:
2026-08-19 15:30:27+00:00 -> 2026-08-19 22:12:18+00:00

All L1-L20 quantity columns found.

Merged shape: (241111, 94)
Total missing L1-L20 values after merge: 0
FULL HFT FEATURE BUILD COMPLETE
Rows: 241,111
Columns: 38

Range:
2026-08-19 15:30:27+00:00 -> 2026-08-19 22:12:18+00:00

Final causal signals:
  signal_standard_microprice
  signal_replenishment_microprice
  signal_hit_hold_asymmetry
  signal_replenishment_asymmetry
  signal_penetration_immediate
  signal_penetration_volume
  signal_penetration_rate_1s
  signal_penetration_momentum
  signal_obi_1
  signal_obi_5
  signal_obi_20
  signal_weighted_obi_20
  signal_trade_pressure

Missing fractions:


,missing_fraction
signal_trade_pressure,0.379526
trade_pressure,0.379522
signal_penetration_momentum,0.000087
penetration_rate_slow,0.000083
penetration_momentum,0.000083
signal_penetration_rate_1s,0.000046
penetration_rate_1s,0.000041
penetration_rate_fast,0.000021
signal_replenishment_microprice,0.000017
replenishment_edge_bps,0.000012



Preview:


,btc_mid_price,btc_spread,standard_microprice,standard_microprice_edge_bps,bid_hit_hold,ask_hit_hold,hit_hold_asymmetry,bid_replenishment,ask_replenishment,replenishment_asymmetry,...,signal_replenishment_asymmetry,signal_penetration_immediate,signal_penetration_volume,signal_penetration_rate_1s,signal_penetration_momentum,signal_obi_1,signal_obi_5,signal_obi_20,signal_weighted_obi_20,signal_trade_pressure
timestamp,,,,,,,,,,,,,,,,,,,,,
2026-08-19 15:30:27+00:00,68005.05,0.1,68005.012211,-0.005557,0.500000,0.500000,0.000000,0.500000,0.500000,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-08-19 15:30:27.100000+00:00,68000.35,0.1,68000.330215,-0.002910,0.500000,0.500000,0.000000,0.500000,0.500000,0.000000,...,0.000000,NaN,NaN,NaN,NaN,-0.755789,-0.596516,0.903110,0.316434,-0.633652
2026-08-19 15:30:27.200000+00:00,68000.05,0.1,68000.094539,0.006550,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,-1.951730,-0.140901,NaN,NaN,-0.395701,0.933485,0.920063,0.761896,0.488101
2026-08-19 15:30:27.300000+00:00,68000.05,0.1,68000.094923,0.006606,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,-2.969586,-2.856755,NaN,NaN,0.890787,0.888610,0.887046,0.890261,-0.897777
2026-08-19 15:30:27.400000+00:00,68000.05,0.1,68000.094941,0.006609,0.033695,0.031203,0.002492,0.000000,0.000000,0.000000,...,0.000000,-0.009039,-0.010150,NaN,NaN,0.898454,0.896255,0.895251,0.897982,-0.986686
2026-08-19 15:30:27.500000+00:00,68000.05,0.1,68000.092749,0.006287,0.065671,0.063140,0.002531,0.010280,0.000000,0.010280,...,0.000000,-0.036368,-0.039351,NaN,NaN,0.898813,0.896551,0.883189,0.897599,-0.991470
2026-08-19 15:30:27.600000+00:00,68000.05,0.1,68000.092336,0.006226,0.096348,0.095053,0.001295,0.009930,0.034064,-0.024134,...,0.010280,-0.000207,-0.000882,NaN,NaN,0.854981,0.852985,0.828391,0.852557,-0.920000
2026-08-19 15:30:27.700000+00:00,68000.05,0.1,68000.087013,0.005443,0.126875,0.125879,0.000996,0.041550,0.066967,-0.025417,...,-0.024134,-0.084865,-0.117523,NaN,NaN,0.846718,0.844724,0.831968,0.845635,-0.954266
2026-08-19 15:30:27.800000+00:00,68000.05,0.1,68000.072240,0.003271,0.152489,0.155655,-0.003165,0.066271,0.098750,-0.032478,...,-0.025417,-0.510463,-0.519840,NaN,NaN,0.740253,0.738530,0.726821,0.739332,-0.996740


In [7]:
research_df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 241111 entries, 2026-08-19 15:30:27+00:00 to 2026-08-19 22:12:18+00:00
Data columns (total 38 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   btc_mid_price                    241111 non-null  float64
 1   btc_spread                       241111 non-null  float64
 2   standard_microprice              241111 non-null  float64
 3   standard_microprice_edge_bps     241111 non-null  float64
 4   bid_hit_hold                     241111 non-null  float64
 5   ask_hit_hold                     241111 non-null  float64
 6   hit_hold_asymmetry               241111 non-null  float64
 7   bid_replenishment                241111 non-null  float64
 8   ask_replenishment                241111 non-null  float64
 9   replenishment_asymmetry          241111 non-null  float64
 10  replenishment_microprice         241108 non-null  float64
 11  replenishment_edge_

In [16]:
# =============================================================================
# MULTI-TIMESCALE 100ms EXPECTED RETURN MODEL
#
# INPUT:
#     research_df
#
# OUTPUT:
#     expected_df
#
# ARCHITECTURE:
#
# microstructure state
#       |
#       +--> 30-minute Ridge model
#       +--> 10-minute Ridge model
#       +--> 1-minute Ridge model
#       +--> 30-second Ridge model
#                         |
#                         v
#                  META REGRESSION
#                  + current factors
#                         |
#                         v
#              expected_return_100ms
#
# Then:
#
# expected return vs actual return
#              |
#              v
#      causal reliability state
#
# =============================================================================


import time
import numpy as np
import pandas as pd

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error


# =============================================================================
# 1. CONFIGURATION
# =============================================================================

# 100ms data
HORIZON_ROWS = 1

# Log return expressed in basis points
RETURN_SCALE = 10_000

# Ridge regularisation
ALPHA = 1.0


# -----------------------------------------------------------------------------
# Historical windows used by the four base models
# -----------------------------------------------------------------------------

WINDOWS = {

    "30m": 30 * 60 * 10,   # 18,000 observations

    "10m": 10 * 60 * 10,   # 6,000 observations

    "1m": 60 * 10,         # 600 observations

    "30s": 30 * 10,        # 300 observations
}


# -----------------------------------------------------------------------------
# Minimum number of valid observations required before a model can train
# -----------------------------------------------------------------------------

MIN_TRAIN = {

    "30m": 3000,

    "10m": 1500,

    "1m": 300,

    "30s": 150,
}


# -----------------------------------------------------------------------------
# HOW OFTEN EACH MODEL IS RETRAINED
#
# Predictions are STILL produced every 100ms.
#
# Only the coefficients are updated at these frequencies.
# -----------------------------------------------------------------------------

UPDATE_FREQUENCY = {

    "30m": 100,    # every 10 seconds

    "10m": 50,     # every 5 seconds

    "1m": 10,      # every 1 second

    "30s": 5,      # every 500 ms
}


# -----------------------------------------------------------------------------
# Meta model
# -----------------------------------------------------------------------------

META_WINDOW = 30 * 60 * 10       # previous 30 minutes
META_MIN_TRAIN = 2000

# Retrain meta model every second
META_UPDATE_EVERY = 10


# =============================================================================
# 2. COPY ORIGINAL DATA
# =============================================================================

TEST_ROWS = 30_000

model_df = (
    research_df
    .iloc[:TEST_ROWS]
    .copy()
    .sort_index()
)

print("TEST MODE")
print("Rows:", len(model_df), flush=True)

N = len(model_df)

print("=" * 90)
print("MULTI-TIMESCALE 100ms EXPECTATION MODEL")
print("=" * 90)
print(f"Rows: {N:,}")
print(f"Start: {model_df.index.min()}")
print(f"End:   {model_df.index.max()}")


# =============================================================================
# 3. MICROPRICE -> RELATIVE EDGE
#
# We do NOT want the absolute BTC price as a feature.
#
# Instead:
#
# replenishment microprice - current mid
#
# expressed in basis points.
# =============================================================================

model_df["replenishment_microprice_edge_bps"] = (

    10_000
    *
    (
        model_df["replenishment_microprice"]
        -
        model_df["btc_mid_price"]
    )
    /
    model_df["btc_mid_price"]

)


# =============================================================================
# 4. FEATURES
# =============================================================================

FEATURES = [

    "replenishment_asymmetry",

    "replenishment_microprice_edge_bps",

    "buy_penetration",

    "sell_penetration",

    "penetration_immediate",

    "penetration_volume",

    "obi_1",

    "obi_5",

    "obi_20",
]


# Check features exist
missing_features = [

    c for c in FEATURES

    if c not in model_df.columns
]

if missing_features:

    raise ValueError(
        f"Missing required features: {missing_features}"
    )


print("\nFeatures used:")

for c in FEATURES:
    print("  ", c)


# =============================================================================
# 5. ACTUAL FUTURE 100ms RETURN
#
# IMPORTANT:
#
# This is the LABEL.
#
# It must NEVER be used as a predictor at the same timestamp.
# =============================================================================

model_df["actual_return_100ms"] = (

    RETURN_SCALE
    *
    np.log(
        model_df["btc_mid_price"].shift(-HORIZON_ROWS)
        /
        model_df["btc_mid_price"]
    )

)


# =============================================================================
# 6. RIDGE MODEL
#
# StandardScaler is inside the pipeline because penetration, OBI,
# replenishment etc. have very different numerical scales.
# =============================================================================

def make_ridge():

    return Pipeline([

        (
            "scaler",
            StandardScaler()
        ),

        (
            "ridge",
            Ridge(alpha=ALPHA)
        )

    ])


# =============================================================================
# 7. PREPARE NUMPY STORAGE
#
# Faster than repeatedly writing into Pandas with .loc
# =============================================================================

base_predictions = {

    name: np.full(
        N,
        np.nan,
        dtype=float
    )

    for name in WINDOWS
}


final_predictions = np.full(
    N,
    np.nan,
    dtype=float
)


# =============================================================================
# 8. DATA USED BY BASE MODELS
# =============================================================================

X_all = model_df[FEATURES]

y_all = model_df["actual_return_100ms"]


# =============================================================================
# 9. WALK-FORWARD BASE MODELS
# =============================================================================

print("\n" + "=" * 90)
print("STAGE 1 — BASE MODELS")
print("=" * 90)

start_time = time.time()


last_models = {

    "30m": None,

    "10m": None,

    "1m": None,

    "30s": None,
}


for t in range(N):

    # -------------------------------------------------------------------------
    # Progress report
    # -------------------------------------------------------------------------

    if t % 10_000 == 0:

        elapsed = time.time() - start_time

        pct = 100 * t / N

        print(
            f"Base models: "
            f"{t:,}/{N:,} "
            f"({pct:.1f}%) "
            f"| elapsed {elapsed/60:.1f} min",
            flush=True
        )


    # -------------------------------------------------------------------------
    # Current state
    # -------------------------------------------------------------------------

    x_current = X_all.iloc[[t]]


    if x_current.isna().any(axis=1).iloc[0]:
        continue


    # -------------------------------------------------------------------------
    # Train / update each timescale independently
    # -------------------------------------------------------------------------

    for name, window in WINDOWS.items():

        update_every = UPDATE_FREQUENCY[name]


        # Only refit when its update clock fires
        if t % update_every == 0:

            # -------------------------------------------------------------
            # At time t:
            #
            # y[t-1] = return from t-1 to t
            #
            # is already known.
            #
            # Therefore t-1 may safely be included in training.
            #
            # Python slicing is end-exclusive, hence:
            #
            # train_end_exclusive = t
            # -------------------------------------------------------------

            train_end_exclusive = t

            if train_end_exclusive <= 0:
                continue


            train_start = max(
                0,
                train_end_exclusive - window
            )


            X_train = X_all.iloc[
                train_start:train_end_exclusive
            ]


            y_train = y_all.iloc[
                train_start:train_end_exclusive
            ]


            valid = (

                X_train.notna().all(axis=1)

                &

                y_train.notna()
            )


            X_train_valid = X_train.loc[valid]

            y_train_valid = y_train.loc[valid]


            if len(X_train_valid) < MIN_TRAIN[name]:
                continue


            model = make_ridge()


            model.fit(
                X_train_valid,
                y_train_valid
            )


            last_models[name] = model


        # ---------------------------------------------------------------------
        # Predict EVERY 100ms using latest available fitted model
        # ---------------------------------------------------------------------

        model = last_models[name]


        if model is None:
            continue


        base_predictions[name][t] = (

            model.predict(
                x_current
            )[0]

        )


# =============================================================================
# 10. ADD BASE PREDICTIONS TO DATAFRAME
# =============================================================================

for name in WINDOWS:

    model_df[
        f"expected_return_100ms_{name}"
    ] = base_predictions[name]


print(
    "\n✅ Base-model stage finished "
    f"in {(time.time() - start_time)/60:.1f} minutes."
)


# =============================================================================
# 11. BASE MODEL PREDICTION COLUMNS
# =============================================================================

BASE_PREDICTIONS = [

    "expected_return_100ms_30m",

    "expected_return_100ms_10m",

    "expected_return_100ms_1m",

    "expected_return_100ms_30s"
]


# =============================================================================
# 12. META MODEL FEATURES
#
# Meta model receives:
#
# 1. Each base model's current expected return
# 2. The actual current microstructure state
#
# Therefore it can learn:
#
# - how much to trust each timescale
# - whether individual order-book factors add information
# =============================================================================

META_FEATURES = (

    BASE_PREDICTIONS
    +
    FEATURES

)


# =============================================================================
# 13. WALK-FORWARD META MODEL
# =============================================================================

print("\n" + "=" * 90)
print("STAGE 2 — META MODEL")
print("=" * 90)

meta_start = time.time()


last_meta_model = None


for t in range(N):

    # -------------------------------------------------------------------------
    # Progress
    # -------------------------------------------------------------------------

    if t % 10_000 == 0:

        elapsed = time.time() - meta_start

        pct = 100 * t / N

        print(
            f"Meta model: "
            f"{t:,}/{N:,} "
            f"({pct:.1f}%) "
            f"| elapsed {elapsed/60:.1f} min",
            flush=True
        )


    # -------------------------------------------------------------------------
    # Current meta state
    # -------------------------------------------------------------------------

    current_meta = model_df[
        META_FEATURES
    ].iloc[[t]]


    if current_meta.isna().any(axis=1).iloc[0]:
        continue


    # -------------------------------------------------------------------------
    # Retrain only every second
    # -------------------------------------------------------------------------

    if t % META_UPDATE_EVERY == 0:

        # y[t-1] is already observable at t
        train_end_exclusive = t

        if train_end_exclusive > 0:

            train_start = max(
                0,
                train_end_exclusive - META_WINDOW
            )


            X_meta_train = model_df[
                META_FEATURES
            ].iloc[
                train_start:train_end_exclusive
            ]


            y_meta_train = model_df[
                "actual_return_100ms"
            ].iloc[
                train_start:train_end_exclusive
            ]


            valid = (

                X_meta_train.notna().all(axis=1)

                &

                y_meta_train.notna()
            )


            X_meta_valid = X_meta_train.loc[
                valid
            ]


            y_meta_valid = y_meta_train.loc[
                valid
            ]


            if len(X_meta_valid) >= META_MIN_TRAIN:

                meta_model = make_ridge()


                meta_model.fit(
                    X_meta_valid,
                    y_meta_valid
                )


                last_meta_model = meta_model


    # -------------------------------------------------------------------------
    # Predict EVERY 100ms
    # -------------------------------------------------------------------------

    if last_meta_model is not None:

        final_predictions[t] = (

            last_meta_model.predict(
                current_meta
            )[0]

        )


# =============================================================================
# 14. STORE FINAL EXPECTED RETURN
# =============================================================================

model_df["expected_return_100ms"] = (
    final_predictions
)


print(
    "\n✅ Meta-model stage finished "
    f"in {(time.time() - meta_start)/60:.1f} minutes."
)


# =============================================================================
# 15. RAW MODEL ERROR
#
# This is only an analytical intermediate variable.
#
# The error at row t cannot be used at row t because actual_return[t]
# occurs in the future.
# =============================================================================

model_df["prediction_error_100ms_raw"] = (

    model_df["actual_return_100ms"]
    -
    model_df["expected_return_100ms"]

)


model_df["abs_prediction_error_100ms_raw"] = (

    model_df[
        "prediction_error_100ms_raw"
    ].abs()

)


# =============================================================================
# 16. DIRECTION MATCH
# =============================================================================

actual_sign = np.sign(
    model_df["actual_return_100ms"]
)


expected_sign = np.sign(
    model_df["expected_return_100ms"]
)


direction_match_raw = (

    actual_sign
    ==
    expected_sign

).astype(float)


# If either value is missing, result should be missing
direction_valid = (

    model_df["actual_return_100ms"].notna()

    &

    model_df["expected_return_100ms"].notna()

)


model_df["direction_match_100ms_raw"] = (

    pd.Series(
        direction_match_raw,
        index=model_df.index
    )
    .where(direction_valid)

)


# =============================================================================
# 17. CAUSAL RELIABILITY STATE
#
# Prediction created at t-1:
#
#     predicted return t-1 -> t
#
# At t:
#
#     that outcome is now known
#
# Therefore we shift the diagnostic forward by one observation.
# =============================================================================

model_df["prediction_error_100ms"] = (

    model_df[
        "prediction_error_100ms_raw"
    ]
    .shift(HORIZON_ROWS)

)


model_df["abs_prediction_error_100ms"] = (

    model_df[
        "abs_prediction_error_100ms_raw"
    ]
    .shift(HORIZON_ROWS)

)


model_df["direction_match_100ms"] = (

    model_df[
        "direction_match_100ms_raw"
    ]
    .shift(HORIZON_ROWS)

)


# =============================================================================
# 18. RELIABILITY — 5 SECOND STATE
# =============================================================================

WINDOW_5S = 50


model_df["rolling_bias_5s"] = (

    model_df[
        "prediction_error_100ms"
    ]
    .rolling(
        WINDOW_5S,
        min_periods=20
    )
    .mean()

)


model_df["rolling_mae_5s"] = (

    model_df[
        "abs_prediction_error_100ms"
    ]
    .rolling(
        WINDOW_5S,
        min_periods=20
    )
    .mean()

)


model_df["rolling_direction_accuracy_5s"] = (

    model_df[
        "direction_match_100ms"
    ]
    .rolling(
        WINDOW_5S,
        min_periods=20
    )
    .mean()

)


# =============================================================================
# 19. RELIABILITY — 30 SECOND STATE
# =============================================================================

WINDOW_30S = 300


model_df["rolling_bias_30s"] = (

    model_df[
        "prediction_error_100ms"
    ]
    .rolling(
        WINDOW_30S,
        min_periods=100
    )
    .mean()

)


model_df["rolling_mae_30s"] = (

    model_df[
        "abs_prediction_error_100ms"
    ]
    .rolling(
        WINDOW_30S,
        min_periods=100
    )
    .mean()

)


model_df["rolling_direction_accuracy_30s"] = (

    model_df[
        "direction_match_100ms"
    ]
    .rolling(
        WINDOW_30S,
        min_periods=100
    )
    .mean()

)


# =============================================================================
# 20. EXPECTATION DISAGREEMENT
#
# How much do our different historical horizons disagree?
#
# High value:
#     different market memories imply very different returns.
#
# Low value:
#     30s / 1m / 10m / 30m models broadly agree.
#
# This becomes another useful uncertainty feature.
# =============================================================================

model_df["timescale_disagreement_100ms"] = (

    model_df[
        BASE_PREDICTIONS
    ]
    .std(
        axis=1
    )

)


# =============================================================================
# 21. EXPECTATION STRENGTH
#
# Absolute size of model's forecast.
#
# Direction:
#     expected_return_100ms
#
# Conviction / magnitude:
#     expected_return_strength_100ms
# =============================================================================

model_df["expected_return_strength_100ms"] = (

    model_df[
        "expected_return_100ms"
    ]
    .abs()

)


# =============================================================================
# 22. FINAL FEATURE TABLE
# =============================================================================

FINAL_FEATURES = [

    # -------------------------------------------------------------------------
    # Original microstructure state
    # -------------------------------------------------------------------------

    "replenishment_asymmetry",

    "replenishment_microprice_edge_bps",

    "buy_penetration",

    "sell_penetration",

    "penetration_immediate",

    "penetration_volume",

    "obi_1",

    "obi_5",

    "obi_20",


    # -------------------------------------------------------------------------
    # Base model expectations
    # -------------------------------------------------------------------------

    "expected_return_100ms_30m",

    "expected_return_100ms_10m",

    "expected_return_100ms_1m",

    "expected_return_100ms_30s",


    # -------------------------------------------------------------------------
    # FINAL MODEL EXPECTATION
    # -------------------------------------------------------------------------

    "expected_return_100ms",

    "expected_return_strength_100ms",

    "timescale_disagreement_100ms",


    # -------------------------------------------------------------------------
    # Causal reliability
    # -------------------------------------------------------------------------

    "prediction_error_100ms",

    "abs_prediction_error_100ms",

    "direction_match_100ms",

    "rolling_bias_5s",

    "rolling_bias_30s",

    "rolling_mae_5s",

    "rolling_mae_30s",

    "rolling_direction_accuracy_5s",

    "rolling_direction_accuracy_30s",


    # -------------------------------------------------------------------------
    # LABEL
    #
    # Do NOT use this as a predictor.
    # -------------------------------------------------------------------------

    "actual_return_100ms"
]


expected_df = model_df[
    FINAL_FEATURES
].copy()


# =============================================================================
# 23. MODEL EVALUATION
# =============================================================================

evaluation = model_df[

    [
        "expected_return_100ms",
        "actual_return_100ms"
    ]

].dropna()


print("\n" + "=" * 90)
print("MODEL EVALUATION")
print("=" * 90)


if len(evaluation) > 0:

    y_true = evaluation[
        "actual_return_100ms"
    ].to_numpy()


    y_pred = evaluation[
        "expected_return_100ms"
    ].to_numpy()


    mae = mean_absolute_error(
        y_true,
        y_pred
    )


    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )


    direction_accuracy = np.mean(

        np.sign(y_true)
        ==
        np.sign(y_pred)

    )


    # Pearson correlation
    if (
        np.std(y_true) > 0
        and
        np.std(y_pred) > 0
    ):

        pearson = np.corrcoef(
            y_true,
            y_pred
        )[0, 1]

    else:

        pearson = np.nan


    # Spearman / rank correlation
    rank_corr = pd.Series(
        y_true
    ).corr(
        pd.Series(y_pred),
        method="spearman"
    )


    print(
        f"Valid predictions:     {len(evaluation):,}"
    )

    print(
        f"MAE:                   {mae:.6f} bps"
    )

    print(
        f"RMSE:                  {rmse:.6f} bps"
    )

    print(
        f"Direction accuracy:    {direction_accuracy:.2%}"
    )

    print(
        f"Pearson correlation:   {pearson:.6f}"
    )

    print(
        f"Spearman correlation:  {rank_corr:.6f}"
    )


else:

    print(
        "No valid final predictions were generated."
    )


# =============================================================================
# 24. EVALUATE EACH BASE MODEL TOO
# =============================================================================

print("\n" + "=" * 90)
print("BASE MODEL COMPARISON")
print("=" * 90)


comparison_rows = []


for col in BASE_PREDICTIONS + [
    "expected_return_100ms"
]:

    temp = model_df[
        [
            col,
            "actual_return_100ms"
        ]
    ].dropna()


    if len(temp) == 0:
        continue


    actual = temp[
        "actual_return_100ms"
    ].to_numpy()


    predicted = temp[
        col
    ].to_numpy()


    comparison_rows.append({

        "model": col,

        "n": len(temp),

        "MAE_bps":
            np.mean(
                np.abs(
                    actual - predicted
                )
            ),

        "RMSE_bps":
            np.sqrt(
                np.mean(
                    (
                        actual - predicted
                    ) ** 2
                )
            ),

        "direction_accuracy":
            np.mean(
                np.sign(actual)
                ==
                np.sign(predicted)
            ),

        "pearson":
            (
                np.corrcoef(
                    actual,
                    predicted
                )[0, 1]
                if
                np.std(actual) > 0
                and
                np.std(predicted) > 0
                else
                np.nan
            ),

        "spearman":
            pd.Series(
                actual
            ).corr(
                pd.Series(predicted),
                method="spearman"
            )
    })


comparison_df = pd.DataFrame(
    comparison_rows
)


if len(comparison_df) > 0:

    display(
        comparison_df
        .sort_values(
            "MAE_bps"
        )
        .reset_index(
            drop=True
        )
    )


# =============================================================================
# 25. MISSING VALUES
# =============================================================================

print("\n" + "=" * 90)
print("FINAL DATAFRAME")
print("=" * 90)

print(
    "Shape:",
    expected_df.shape
)


print("\nMissing fractions:")

display(

    expected_df
    .isna()
    .mean()
    .sort_values(
        ascending=False
    )
    .to_frame(
        "missing_fraction"
    )

)


# =============================================================================
# 26. LAST 20 OBSERVATIONS
# =============================================================================

print("\nLast 20 rows:")

display(
    expected_df.tail(20)
)


# =============================================================================
# 27. MOST IMPORTANT OUTPUTS
# =============================================================================

print("\n" + "=" * 90)
print("KEY NEW FEATURES")
print("=" * 90)

print("""
expected_return_100ms_30m
    -> expected next-100ms return according to the previous 30-minute regime

expected_return_100ms_10m
    -> expected next-100ms return according to the previous 10-minute regime

expected_return_100ms_1m
    -> expected next-100ms return according to the previous 1-minute regime

expected_return_100ms_30s
    -> expected next-100ms return according to the previous 30-second regime

expected_return_100ms
    -> FINAL expected 100ms return from the meta model

expected_return_strength_100ms
    -> absolute magnitude of the final expectation

timescale_disagreement_100ms
    -> disagreement between 30m / 10m / 1m / 30s expectations

prediction_error_100ms
    -> previous matured prediction error: actual - expected

rolling_bias_5s / rolling_bias_30s
    -> whether the model has recently underestimated or overestimated returns

rolling_mae_5s / rolling_mae_30s
    -> how inaccurate the model has recently been

rolling_direction_accuracy_5s / 30s
    -> how often recent predictions got the direction correct

actual_return_100ms
    -> TARGET / evaluation label only
""")

print("=" * 90)
print("✅ EVERYTHING COMPLETE")
print("=" * 90)

TEST MODE
Rows: 30000
MULTI-TIMESCALE 100ms EXPECTATION MODEL
Rows: 30,000
Start: 2026-08-19 15:30:27+00:00
End:   2026-08-19 16:20:26.900000+00:00

Features used:
   replenishment_asymmetry
   replenishment_microprice_edge_bps
   buy_penetration
   sell_penetration
   penetration_immediate
   penetration_volume
   obi_1
   obi_5
   obi_20

STAGE 1 — BASE MODELS
Base models: 0/30,000 (0.0%) | elapsed 0.0 min
Base models: 10,000/30,000 (33.3%) | elapsed 1.3 min
Base models: 20,000/30,000 (66.7%) | elapsed 2.8 min

✅ Base-model stage finished in 4.3 minutes.

STAGE 2 — META MODEL
Meta model: 0/30,000 (0.0%) | elapsed 0.0 min
Meta model: 10,000/30,000 (33.3%) | elapsed 0.6 min
Meta model: 20,000/30,000 (66.7%) | elapsed 1.4 min

✅ Meta-model stage finished in 2.3 minutes.

MODEL EVALUATION
Valid predictions:     24,899
MAE:                   0.216517 bps
RMSE:                  0.428945 bps
Direction accuracy:    20.97%
Pearson correlation:   0.430221
Spearman correlation:  0.461953

BASE 

,model,n,MAE_bps,RMSE_bps,direction_accuracy,pearson,spearman
0,expected_return_100ms,24899,0.216517,0.428945,0.209727,0.430221,0.461953
1,expected_return_100ms_10m,28449,0.251804,0.479760,0.233260,0.450977,0.467112
2,expected_return_100ms_30m,26899,0.254202,0.458620,0.222090,0.441632,0.464323
3,expected_return_100ms_1m,29689,0.267943,0.558822,0.241032,0.350419,0.455939
4,expected_return_100ms_30s,29844,0.278143,0.659698,0.240048,0.254743,0.443977



FINAL DATAFRAME
Shape: (30000, 26)

Missing fractions:


,missing_fraction
rolling_direction_accuracy_30s,0.173333
rolling_mae_30s,0.173333
rolling_bias_30s,0.173333
rolling_bias_5s,0.170667
rolling_direction_accuracy_5s,0.170667
rolling_mae_5s,0.170667
abs_prediction_error_100ms,0.170033
prediction_error_100ms,0.170033
direction_match_100ms,0.170033
expected_return_100ms,0.170000



Last 20 rows:


,replenishment_asymmetry,replenishment_microprice_edge_bps,buy_penetration,sell_penetration,penetration_immediate,penetration_volume,obi_1,obi_5,obi_20,expected_return_100ms_30m,...,prediction_error_100ms,abs_prediction_error_100ms,direction_match_100ms,rolling_bias_5s,rolling_bias_30s,rolling_mae_5s,rolling_mae_30s,rolling_direction_accuracy_5s,rolling_direction_accuracy_30s,actual_return_100ms
timestamp,,,,,,,,,,,,,,,,,,,,,
2026-08-19 16:20:25+00:00,0.032609,-0.002286,0.000000,0.000000,0.000000,0.000000,-0.341228,-0.342576,-0.488335,-0.054744,...,0.050922,0.050922,0.0,0.037134,0.020733,0.063236,0.113048,0.04,0.090000,0.000000
2026-08-19 16:20:25.100000+00:00,0.032609,-0.002254,0.003724,0.018831,-0.015107,-0.012357,-0.336955,-0.337425,-0.481860,-0.054383,...,0.039875,0.039875,0.0,0.038075,0.020692,0.063890,0.113008,0.04,0.090000,0.000000
2026-08-19 16:20:25.200000+00:00,0.065562,-0.002105,0.000000,0.000000,0.000000,0.000000,-0.344289,-0.344735,-0.488425,-0.055247,...,0.039683,0.039683,0.0,0.038900,0.020647,0.064653,0.112963,0.04,0.090000,0.000000
2026-08-19 16:20:25.300000+00:00,0.065562,-0.002104,0.000000,0.018204,-0.018204,-0.018204,-0.344123,-0.344569,-0.488270,-0.055751,...,0.040022,0.040022,0.0,0.039700,0.020616,0.065453,0.112932,0.04,0.090000,0.000000
2026-08-19 16:20:25.400000+00:00,0.080245,-0.003887,0.004378,0.000000,0.004378,0.004378,-0.584833,-0.584251,-0.715601,-0.085234,...,0.040521,0.040521,0.0,0.040594,0.020584,0.066180,0.112900,0.04,0.090000,0.000000
2026-08-19 16:20:25.500000+00:00,0.063329,-0.002586,0.014817,0.061660,-0.046843,-0.025071,-0.404480,-0.404450,-0.533635,-0.062438,...,0.063767,0.063767,0.0,0.042139,0.020634,0.067187,0.112950,0.04,0.090000,0.000000
2026-08-19 16:20:25.600000+00:00,0.061172,-0.002671,0.002068,0.008472,-0.006404,-0.004621,-0.412729,-0.413584,-0.542635,-0.063083,...,0.045769,0.045769,0.0,0.043499,0.020624,0.067657,0.112940,0.04,0.090000,0.000000
2026-08-19 16:20:25.700000+00:00,0.059088,-0.002321,0.003317,0.000000,0.003317,0.003317,-0.366517,-0.367807,-0.516556,-0.058371,...,0.046129,0.046129,0.0,0.045575,0.020616,0.067426,0.112931,0.04,0.090000,0.000000
2026-08-19 16:20:25.800000+00:00,0.076746,-0.002204,0.003729,0.000268,0.003461,0.003601,-0.366202,-0.367495,-0.516279,-0.058490,...,0.042251,0.042251,0.0,0.037704,0.020594,0.059555,0.112910,0.02,0.090000,0.000000



KEY NEW FEATURES

expected_return_100ms_30m
    -> expected next-100ms return according to the previous 30-minute regime

expected_return_100ms_10m
    -> expected next-100ms return according to the previous 10-minute regime

expected_return_100ms_1m
    -> expected next-100ms return according to the previous 1-minute regime

expected_return_100ms_30s
    -> expected next-100ms return according to the previous 30-second regime

expected_return_100ms
    -> FINAL expected 100ms return from the meta model

expected_return_strength_100ms
    -> absolute magnitude of the final expectation

timescale_disagreement_100ms
    -> disagreement between 30m / 10m / 1m / 30s expectations

prediction_error_100ms
    -> previous matured prediction error: actual - expected

rolling_bias_5s / rolling_bias_30s
    -> whether the model has recently underestimated or overestimated returns

rolling_mae_5s / rolling_mae_30s
    -> how inaccurate the model has recently been

rolling_direction_accuracy_5s / 

In [17]:
# =============================================================================
# BETTER DIAGNOSTICS FOR 100ms EXPECTATION MODEL
# =============================================================================

import numpy as np
import pandas as pd


eval_df = model_df[
    [
        "expected_return_100ms",
        "actual_return_100ms"
    ]
].dropna().copy()


pred = eval_df["expected_return_100ms"]
actual = eval_df["actual_return_100ms"]


print("=" * 90)
print("100ms MODEL — DETAILED DIAGNOSTICS")
print("=" * 90)


# =============================================================================
# 1. HOW OFTEN DOES PRICE ACTUALLY MOVE?
# =============================================================================

zero_rate = (actual == 0).mean()

nonzero_rate = (actual != 0).mean()

print(f"\nZero-return observations:     {zero_rate:.2%}")
print(f"Non-zero-return observations: {nonzero_rate:.2%}")


# =============================================================================
# 2. ZERO-PREDICTION BASELINE
#
# If we always predicted 0 bps, what MAE would we get?
#
# This is extremely important because many 100ms returns are zero.
# =============================================================================

zero_baseline_mae = np.mean(
    np.abs(actual)
)

model_mae = np.mean(
    np.abs(actual - pred)
)

improvement = (
    1
    -
    model_mae / zero_baseline_mae
)

print("\n--- MAE ---")

print(
    f"Zero baseline MAE: {zero_baseline_mae:.6f} bps"
)

print(
    f"Model MAE:         {model_mae:.6f} bps"
)

print(
    f"MAE improvement:   {improvement:.2%}"
)


# =============================================================================
# 3. DIRECTION ACCURACY ONLY WHEN PRICE MOVED
# =============================================================================

moving = actual != 0

direction_accuracy_nonzero = np.mean(
    np.sign(actual[moving])
    ==
    np.sign(pred[moving])
)

print("\n--- DIRECTION ---")

print(
    f"Direction accuracy when return != 0: "
    f"{direction_accuracy_nonzero:.2%}"
)


# =============================================================================
# 4. CONDITIONAL CORRELATION WHEN PRICE MOVES
# =============================================================================

if moving.sum() > 2:

    pearson_nonzero = actual[moving].corr(
        pred[moving],
        method="pearson"
    )

    spearman_nonzero = actual[moving].corr(
        pred[moving],
        method="spearman"
    )

else:

    pearson_nonzero = np.nan
    spearman_nonzero = np.nan


print(
    f"Pearson | moving observations:  "
    f"{pearson_nonzero:.4f}"
)

print(
    f"Spearman | moving observations: "
    f"{spearman_nonzero:.4f}"
)


# =============================================================================
# 5. EXPECTATION STRENGTH
#
# Does a stronger expected return correspond to larger actual moves?
# =============================================================================

temp = eval_df.copy()

temp["prediction_strength"] = (
    temp["expected_return_100ms"].abs()
)

temp["actual_abs_return"] = (
    temp["actual_return_100ms"].abs()
)


temp["strength_decile"] = pd.qcut(
    temp["prediction_strength"],
    10,
    labels=False,
    duplicates="drop"
)


strength_table = (
    temp
    .groupby("strength_decile")
    .agg(
        n=("actual_return_100ms", "size"),

        mean_prediction_strength=(
            "prediction_strength",
            "mean"
        ),

        mean_actual_abs_return=(
            "actual_abs_return",
            "mean"
        ),

        mean_expected_return=(
            "expected_return_100ms",
            "mean"
        ),

        mean_actual_return=(
            "actual_return_100ms",
            "mean"
        )
    )
)


print("\n--- PREDICTION STRENGTH DECILES ---")

display(strength_table)


# =============================================================================
# 6. EXPECTED RETURN DECILES
#
# This is one of the most important tests.
#
# If the model contains useful information:
#
# lowest predicted-return deciles should have lower realized returns
# highest predicted-return deciles should have higher realized returns.
# =============================================================================

temp["prediction_decile"] = pd.qcut(
    temp["expected_return_100ms"],
    10,
    labels=False,
    duplicates="drop"
)


decile_table = (
    temp
    .groupby("prediction_decile")
    .agg(
        n=("actual_return_100ms", "size"),

        mean_expected_return=(
            "expected_return_100ms",
            "mean"
        ),

        mean_actual_return=(
            "actual_return_100ms",
            "mean"
        ),

        median_actual_return=(
            "actual_return_100ms",
            "median"
        ),

        probability_up=(
            "actual_return_100ms",
            lambda x: (x > 0).mean()
        ),

        probability_down=(
            "actual_return_100ms",
            lambda x: (x < 0).mean()
        )
    )
)


print("\n--- EXPECTED RETURN DECILES ---")

display(decile_table)


# =============================================================================
# 7. EXTREME EXPECTATIONS
#
# Often an HFT feature is most useful at extremes rather than every 100ms.
# =============================================================================

q10 = pred.quantile(0.10)
q90 = pred.quantile(0.90)

bottom = eval_df[
    pred <= q10
]

top = eval_df[
    pred >= q90
]


print("\n--- EXTREME SIGNALS ---")

print(
    f"Bottom 10% expected return: "
    f"{bottom['actual_return_100ms'].mean():.6f} bps actual"
)

print(
    f"Top 10% expected return:    "
    f"{top['actual_return_100ms'].mean():.6f} bps actual"
)

print(
    f"Top-minus-bottom spread:    "
    f"{top['actual_return_100ms'].mean() - bottom['actual_return_100ms'].mean():.6f} bps"
)

100ms MODEL — DETAILED DIAGNOSTICS

Zero-return observations:     74.94%
Non-zero-return observations: 25.06%

--- MAE ---
Zero baseline MAE: 0.177483 bps
Model MAE:         0.216517 bps
MAE improvement:   -21.99%

--- DIRECTION ---
Direction accuracy when return != 0: 83.70%
Pearson | moving observations:  0.5189
Spearman | moving observations: 0.6308

--- PREDICTION STRENGTH DECILES ---


,n,mean_prediction_strength,mean_actual_abs_return,mean_expected_return,mean_actual_return
strength_decile,,,,,
0,2490,0.007697,0.050499,0.000034,-0.000903
1,2490,0.023082,0.075557,-0.000510,-0.001089
2,2490,0.039387,0.068976,-0.000813,0.003569
3,2490,0.056073,0.071149,-0.000150,-0.004437
4,2490,0.075081,0.090404,0.000506,0.005422
5,2489,0.095888,0.099396,0.003748,-0.002920
6,2490,0.123677,0.139316,-0.009202,-0.009645
7,2490,0.166544,0.216683,-0.004953,-0.007545
8,2490,0.254839,0.329103,-0.007919,0.041526



--- EXPECTED RETURN DECILES ---


,n,mean_expected_return,mean_actual_return,median_actual_return,probability_up,probability_down
prediction_decile,,,,,,
0,2490,-0.416460,-0.380904,-0.072877,0.055422,0.532530
1,2490,-0.150125,-0.130422,0.000000,0.038956,0.254217
2,2490,-0.089316,-0.056016,0.000000,0.028514,0.129317
3,2490,-0.050486,-0.020908,0.000000,0.038956,0.073494
4,2490,-0.018373,-0.007542,0.000000,0.038554,0.056225
5,2489,0.012284,0.003371,0.000000,0.048614,0.047409
6,2490,0.044942,0.016696,0.000000,0.067068,0.036546
7,2490,0.081986,0.049203,0.000000,0.118876,0.030924
8,2490,0.139821,0.112983,0.000000,0.252610,0.037349



--- EXTREME SIGNALS ---
Bottom 10% expected return: -0.380904 bps actual
Top 10% expected return:    0.429068 bps actual
Top-minus-bottom spread:    0.809972 bps


In [10]:
print("Kernel is alive")

print(research_df.shape)

print(research_df[
    [
        "replenishment_asymmetry",
        "replenishment_microprice",
        "buy_penetration",
        "sell_penetration",
        "penetration_immediate",
        "penetration_volume",
        "obi_1",
        "obi_5",
        "obi_20",
    ]
].head())

Kernel is alive
(241111, 38)
                                  replenishment_asymmetry  \
timestamp                                                   
2026-08-19 15:30:27+00:00                             0.0   
2026-08-19 15:30:27.100000+00:00                      0.0   
2026-08-19 15:30:27.200000+00:00                      0.0   
2026-08-19 15:30:27.300000+00:00                      0.0   
2026-08-19 15:30:27.400000+00:00                      0.0   

                                  replenishment_microprice  buy_penetration  \
timestamp                                                                     
2026-08-19 15:30:27+00:00                     68005.012211              NaN   
2026-08-19 15:30:27.100000+00:00              68000.330215         0.734772   
2026-08-19 15:30:27.200000+00:00                       NaN         0.043384   
2026-08-19 15:30:27.300000+00:00                       NaN         0.001186   
2026-08-19 15:30:27.400000+00:00                       NaN         0.

In [11]:
import time

start = time.time()

test = research_df.iloc[:10000].copy()

print("10k rows copied")
print("Time:", time.time() - start)

10k rows copied
Time: 0.014880895614624023
